# Searching

This section develops search methods from foundational to efficient, then evaluates them with correctness checks and timing evidence.

**Learning Goals**
- Implement linear and binary search correctly
- Explain when sorted order changes what is possible
- Use hash-based lookup for repeated membership checks
- Compare asymptotic cost with measured runtime
- Select a search strategy from data and workload constraints

**Section Roadmap**
- Establish correctness and edge-case behavior first.
- Compare scaling behavior as input size and lookup count increase.
- Use the decision guide to choose an approach for concrete scenarios.

In [ ]:
from time import perf_counter  # High-resolution timer for benchmark cells.
import random  # Randomized sample generation for lookup demos.

## Linear Search

Linear search is the baseline strategy: scan from left to right until the target is found or the input is exhausted.

Because it does not assume any order, it works on both sorted and unsorted data. This universality is its key strength.

| Case | Complexity | When it occurs |
|---|---|---|
| Best | $O(1)$ | Target is the first element |
| Average | $O(n)$ | Target is in the middle region |
| Worst | $O(n)$ | Target is the last element or absent |

**Key properties**
- No preprocessing is required.
- Runtime is proportional to the number of inspected elements.
- It serves as a reference point when comparing faster methods.

In [19]:
def linear_search(items, target):
    """Return index of target in items, or -1 if not found."""
    for i, value in enumerate(items):  # Scan left-to-right until a match is found.
        if value == target:
            return i
    return -1  # -1 signals that target is absent.

sample = [7, 4, 9, 1, 3]  # Quick sanity checks: one present and one absent target.
print(linear_search(sample, 9))   # 2
print(linear_search(sample, 8))   # -1

2
-1


In [ ]:
### Exercise: Find Minimum Value
#   1. Write `min_value(nums)` that scans through the list in a single pass.
#   2. Return `None` for an empty list.
#   3. Return the smallest value otherwise (do not use the built-in `min`).
### Your code starts here.
# Suggested structure:
# - guard clause for empty input
# - initialize current minimum from first element
# - update while scanning remaining elements



### Your code ends here.

In [ ]:
### Solution

def min_value(nums):
    if not nums:  # Guard clause: empty input has no minimum value.
        return None

    current_min = nums[0]  # Initialize once, then refine in one scan.
    for x in nums[1:]:
        if x < current_min:
            current_min = x
    return current_min

> **Next:** If data can be kept sorted and repeated lookups are needed, binary search is usually the better choice.

## Binary Search

Binary search uses **sorted order** to remove half of the remaining candidates at each step.

The algorithm maintains an interval `[left, right]`. At each iteration, it checks the midpoint `mid`:
- if `items[mid] == target`, the search is complete
- if `items[mid] < target`, discard the left half (`left = mid + 1`)
- if `items[mid] > target`, discard the right half (`right = mid - 1`)

This repeated halving yields $O(\log n)$ time. For large inputs, this is dramatically more efficient than a linear scan.

**Why sorted input is required:** midpoint comparisons are only informative when elements are in nondecreasing order.

**Index semantics:** `binary_search()` returns an index into the *sorted* input passed to it, not into the original unsorted sequence.

| Case | Complexity |
|---|---|
| Best | $O(1)$ — target is midpoint on first check |
| Worst / Average | $O(\log n)$ (as developed in 10.1) |

### Iterative Implementation

Read the code below as a direct implementation of the interval idea:

1. Start with the full search interval (`left = 0`, `right = n-1`).
2. Recompute the midpoint each iteration.
3. Keep only the half that can still contain the target.
4. Stop when the interval becomes empty (`left > right`).

**What to observe while reading the code**
- The interval shrinks every iteration, guaranteeing termination.
- Every update preserves correctness: no viable target position is discarded.
- Returning `-1` means the algorithm has proved absence, not just failed to guess.

In [20]:
def binary_search(sorted_items, target):
    """Return index of target in sorted_items, or -1 if not found.

    Note: the returned index is a position in *sorted_items*, not the original
    unsorted list. Use linear_search when you need the original-list position.
    """
    left, right = 0, len(sorted_items) - 1

    while left <= right:
        mid = (left + right) // 2  # Midpoint partitions remaining interval.
        guess = sorted_items[mid]

        if guess == target:
            return mid
        if guess < target:
            left = mid + 1  # Target can only be in the right half.
        else:
            right = mid - 1  # Target can only be in the left half.

    return -1  # Interval exhausted: target is absent.

sorted_sample = [1, 3, 4, 7, 9, 14, 20]
print(binary_search(sorted_sample, 14))  # 5
print(binary_search(sorted_sample, 2))   # -1

5
-1


### Recursive Binary Search (Optional Extension)

A recursive implementation expresses the same halving logic with function calls instead of a loop.

- Time complexity remains $O(\log n)$, the same as the iterative version.
- Extra space differs: recursion uses $O(\log n)$ call-stack space, while iteration uses $O(1)$.

In Python, iteration is usually preferred in production because recursive calls add overhead and are limited by recursion depth. This extension is primarily for conceptual comparison.

In [ ]:
def binary_search_recursive(sorted_items, target, left=0, right=None):
    if right is None:
        right = len(sorted_items) - 1  # Initialize right boundary on first call.

    if left > right:
        return -1  # Base case: empty interval means target is absent.

    mid = (left + right) // 2
    guess = sorted_items[mid]

    if guess == target:
        return mid
    if guess < target:
        return binary_search_recursive(sorted_items, target, mid + 1, right)  # Recur right.
    return binary_search_recursive(sorted_items, target, left, mid - 1)  # Recur left.


sample_sorted = [1, 3, 4, 7, 9, 14, 20]
print('iterative 14:', binary_search(sample_sorted, 14))
print('recursive 14:', binary_search_recursive(sample_sorted, 14))
print('iterative 2 :', binary_search(sample_sorted, 2))
print('recursive 2 :', binary_search_recursive(sample_sorted, 2))

iterative 14: 5
recursive 14: 5
iterative 2 : -1
recursive 2 : -1


## The `bisect` Module

The `bisect` module provides production-ready tools for searching and insertion in sorted lists.

| Function | What it does |
|---|---|
| `bisect_left(a, x)` | Leftmost index where `x` can be inserted while preserving sort order |
| `bisect_right(a, x)` | Rightmost insertion index (also available as `bisect.bisect`) |
| `insort(a, x)` | Inserts `x` into its sorted position (in-place) |

The most important practical pattern is: **locate with `bisect_left`, then verify equality**. This prevents false positives when the target is absent and avoids re-implementing binary search logic.

> **Note:** `insort` is useful for incremental updates. For large batches, it is usually better to append and sort once.

In [ ]:
import bisect

data = [1, 3, 4, 7, 9, 14, 20]

idx = bisect.bisect_left(data, 7)  # 1) Locate an existing value directly.
print(f"bisect_left(7)  → index {idx}, value {data[idx]}")   # 3, value 7

print(f"bisect_left(5)  → index {bisect.bisect_left(data, 5)}")   # 2) Missing value insertion index.
print(f"bisect_right(5) → index {bisect.bisect_right(data, 5)}")  # 3 (same when absent).

bisect.insort(data, 5)  # 3) Insert while preserving sorted order.
print(f"after insort(5): {data}")

def bisect_search(sorted_items, target):  # Standard idiom: locate insertion point, then verify.
    idx = bisect.bisect_left(sorted_items, target)
    if idx < len(sorted_items) and sorted_items[idx] == target:
        return idx
    return -1

nums = [1, 3, 4, 7, 9, 14, 20]
print(bisect_search(nums, 9))   # 4
print(bisect_search(nums, 6))   # -1


bisect_left(7)  → index 3, value 7
bisect_left(5)  → index 3
bisect_right(5) → index 3
after insort(5): [1, 3, 4, 5, 7, 9, 14, 20]
4
-1


## Hash-Based Lookup

When the task is membership testing, a `set` or `dict` is often the most effective tool.

Average lookup is $O(1)$, independent of order, because hash tables use key-based addressing rather than scanning.

**Connection to Chapter 08**
- You previously measured list versus set/dict membership behavior.
- Here, that same $O(1)$ idea is compared directly with linear ($O(n)$) and binary ($O(\log n)$) search.

**Key trade-off:** building the hash table has an upfront $O(n)$ cost, but repeated lookups are fast.

In [ ]:
random.seed(42)  # Fixed seed keeps the demonstration reproducible across runs.
numbers = [random.randint(1, 20_000) for _ in range(15_000)]
lookup_set = set(numbers)

target = numbers[len(numbers) // 2]  # Use one present value for contrast.
missing = 999_999  # Use one absent value for contrast.

print('target in list:', target in numbers)
print('target in set :', target in lookup_set)
print('missing in list:', missing in numbers)
print('missing in set :', missing in lookup_set)

target in list: True
target in set : True
missing in list: False
missing in set : False


## Correctness Checks

Before benchmarking, verify correctness on edge cases that often reveal hidden bugs:

- empty input
- one-element input (present and absent targets)
- duplicates (index semantics matter)
- a target near the end (worst case for linear search)

Using `assert` turns expectations into executable checks. If any assumption is violated, execution stops immediately and the failing case is exposed.

**Contract reminder:** `binary_search()` guarantees a matching index when present, while `binary_search_first()` strengthens that guarantee to the leftmost matching index.

In [ ]:
tests = [  # Each pair is (input_sequence, target).
    ([], 5),
    ([10], 10),
    ([10], 5),
    ([1, 2, 2, 2, 3], 2),
    (list(range(100)), 99),
]

for arr, target in tests:
    lin_idx = linear_search(arr, target)
    arr_sorted = sorted(arr)
    bin_idx = binary_search(arr_sorted, target)

    assert (target in set(arr)) == (target in arr)  # Set/list membership should agree as booleans.

    if target in arr:
        assert lin_idx != -1
        assert arr[lin_idx] == target  # Found case: index must point to a matching value.
        assert bin_idx != -1
        assert arr_sorted[bin_idx] == target
    else:
        assert lin_idx == -1  # Absent case: both searches should report -1.
        assert bin_idx == -1

print('All checks passed.')

All checks passed.


## Timing Comparison

This section evaluates a practical question: **When is preprocessing worth it?**

Two workloads are benchmarked to represent common usage patterns:

- **One lookup:** measure the full workflow cost, including any preprocessing (sorting or building a `set`).
- **Many repeated lookups:** perform preprocessing once, then reuse the resulting structure across many queries.

This side-by-side design separates two ideas that are often conflated:

- **Asymptotic lookup cost** (for example, $O(\log n)$ or $O(1)$), and
- **Total workflow cost** (setup + lookup).

Interpretation guide:

- If only one lookup is needed, preprocessing may dominate total time.
- If many lookups are needed, preprocessing is often amortized, so faster lookup methods become more beneficial.

**Why wrapper functions are used.** `linear_search` and `binary_search` are passed directly to `average_runtime` because their signatures match `fn(data, target)`. Set membership (`target in lookup_set`) is an expression, not a callable with that signature, so it is wrapped in named functions. This keeps the benchmark interface consistent and makes each timing represent a comparable workflow.

We use `time.perf_counter()` for higher-resolution timing. The coarser `time.time()` from earlier chapters is acceptable for rough timing, but less reliable for short operations.

In [21]:
def average_runtime(fn, *args, repeats=30):
    total = 0.0
    for _ in range(repeats):
        start = perf_counter()
        fn(*args)
        total += perf_counter() - start
    return total / repeats  # Average repeated measurements to reduce timing noise.


def sort_then_binary_once(data, target):
    sorted_data = sorted(data)
    return binary_search(sorted_data, target)  # One-time workflow: sort, then one binary lookup.


def build_set_once(data, target):
    lookup_set = set(data)
    return target in lookup_set  # One-time workflow: build hash table, then one membership test.


def linear_many(data, targets):
    return sum(linear_search(data, target) != -1 for target in targets)  # Repeated lookups, no preprocessing.


def sort_then_binary_many(data, targets):
    sorted_data = sorted(data)
    return sum(binary_search(sorted_data, target) != -1 for target in targets)  # Preprocess once, reuse sorted data.


def build_set_many(data, targets):
    lookup_set = set(data)
    return sum(target in lookup_set for target in targets)  # Preprocess once, reuse hash table.


sizes = [10_000, 50_000, 100_000, 200_000]
print('One lookup: includes preprocessing cost')
print(f"{'n':>10} {'linear total':>15} {'sort+binary':>15} {'build+set':>15}")

for n in sizes:
    data = list(range(n))
    target = n - 1

    t_linear_one = average_runtime(linear_search, data, target)
    t_binary_one = average_runtime(sort_then_binary_once, data, target)
    t_set_one = average_runtime(build_set_once, data, target)

    print(f"{n:>10} {t_linear_one:>15.6f} {t_binary_one:>15.6f} {t_set_one:>15.6f}")

print()
print('Many lookups: preprocessing is paid once and reused')
print(f"{'n':>10} {'linear total':>15} {'sort+binary':>15} {'build+set':>15}")

for n in sizes:
    data = list(range(n))
    targets = [0, n // 4, n // 2, (3 * n) // 4, n - 1] * 20  # Mix positions to avoid one-case bias.

    t_linear_many = average_runtime(linear_many, data, targets)
    t_binary_many = average_runtime(sort_then_binary_many, data, targets)
    t_set_many = average_runtime(build_set_many, data, targets)

    print(f"{n:>10} {t_linear_many:>15.6f} {t_binary_many:>15.6f} {t_set_many:>15.6f}")

One lookup: includes preprocessing cost
         n    linear total     sort+binary       build+set
     10000        0.000155        0.000031        0.000049
     50000        0.000768        0.000161        0.000239
    100000        0.001534        0.000305        0.000528
    200000        0.003023        0.000645        0.001115

Many lookups: preprocessing is paid once and reused
         n    linear total     sort+binary       build+set
     10000        0.007685        0.000117        0.000051
     50000        0.038049        0.000246        0.000241
    100000        0.076500        0.000440        0.000527
    200000        0.154864        0.000768        0.001137


In [ ]:
### Exercise: Binary Search First
#   1. Implement `binary_search_first(sorted_items, target)`.
#   2. Return the first (leftmost) index of `target` when duplicates exist.
#   3. Return `-1` when the target is absent.
#   4. Validate with at least: empty input, all duplicates, target missing.
#
#   Hint: when a match is found, do not return immediately. Instead, record
#   the index and continue searching the left half — the leftmost occurrence
#   may be further left. Return the recorded index only after the loop ends.
### Your code starts here.
# Suggested checkpoints:
# - keep a `result` variable initialized to -1
# - on equality, update `result` and move `right` left
# - after loop, return `result`



### Your code ends here.

In [ ]:
### Solution

def binary_search_first(sorted_items, target):
    left, right = 0, len(sorted_items) - 1
    result = -1

    while left <= right:
        mid = (left + right) // 2
        guess = sorted_items[mid]

        if guess == target:
            result = mid  # Record candidate and keep searching left for first occurrence.
            right = mid - 1
        elif guess < target:
            left = mid + 1
        else:
            right = mid - 1

    return result

dup_data = [1, 2, 2, 2, 4, 4, 5]
assert binary_search_first([], 2) == -1  # Empty input check.
assert binary_search_first([2, 2, 2], 2) == 0  # All-duplicate input check.
assert binary_search_first(dup_data, 2) == 1
assert binary_search_first(dup_data, 4) == 4
assert binary_search_first(dup_data, 3) == -1  # Absent target check.

print(binary_search_first(dup_data, 2))  # expected: 1
print(binary_search_first(dup_data, 4))  # expected: 4
print(binary_search_first(dup_data, 3))  # expected: -1
print('binary_search_first checks passed.')

1
4
-1
binary_search_first checks passed.


## Search Strategy Summary

No single search method is optimal in every context. Selection should be based on workload, data properties, and total cost.

| Strategy | Time | Needs sorted data? | Best for |
|---|---|---|---|
| Linear search | $O(n)$ | No | One-off lookups; small or unsorted input |
| Binary search | $O(\log n)$ | Yes | Repeated lookups on sorted data |
| `bisect` module | $O(\log n)$ | Yes | Production search/insertion on sorted lists |
| `set` / `dict` lookup | $O(1)$ average | No | High-volume membership checks |

**Cost of sorting.** Binary search and `bisect` require sorted input. Since sorting costs $O(n \log n)$, it may not pay off for only one or two lookups.

**Preprocessing trade-off.** Building a `set` costs $O(n)$ time and $O(n)$ space, but each later membership check is typically $O(1)$.

**Decision checklist**
- Is the data already sorted?
- How many lookups will be performed?
- Is insertion while preserving order required?
- Is extra memory acceptable?

In practice, choose the method with the lowest **total workflow cost** (setup + lookup), not just the lowest isolated lookup complexity.